# บทที่ 17 - การสร้างเอเจนต์ AI ท้องถิ่นด้วย Foundry Local และ Qwen

ในสมุดบันทึกนี้คุณจะสร้าง **ผู้ช่วยวิศวกรรมท้องถิ่น** ที่ทำงานได้อย่างสมบูรณ์บนเครื่องคอมพิวเตอร์ของคุณเอง โดยไม่ใช้การประมวลผลบนคลาวด์ในทุกขั้นตอน ผู้ช่วยนี้สามารถ:

1. **เรียกใช้เครื่องมือ** ผ่านการเรียกฟังก์ชัน Qwen ผ่าน Foundry Local
2. **แสดงรายการและอ่านไฟล์** ภายในไดเรกทอรีโครงการแบบแซนด์บ็อกซ์
3. **วิเคราะห์โค้ด** ด้วยเมตริกง่าย ๆ
4. **ค้นหาคู่มือ** ด้วย RAG ท้องถิ่น (Chroma)
5. **ใช้เซิร์ฟเวอร์ MCP ท้องถิ่น** (จะข้ามไปอย่างสุภาพถ้าไม่มีการตั้งค่า)

โค้ดเอเจนต์นี้ดูเหมือนกับบทเรียนบนคลาวด์เกือบทั้งหมด — เพียงแต่เปลี่ยนจุดเชื่อมต่อไคลเอนต์จากคลาวด์เป็น `localhost`


## การตั้งค่า

ก่อนที่จะรันโน้ตบุ๊กนี้:

1. **ติดตั้ง Microsoft Foundry Local** (ดู [เอกสาร](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) สำหรับระบบปฏิบัติการของคุณ)
2. **ดาวน์โหลดและเริ่มต้นโมเดล Qwen:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. ติดตั้งแพ็กเกจ Python ด้านล่าง

ทุกอย่างทำงานบนเครื่องท้องถิ่น เครื่องที่มี RAM ประมาณ 8 GB ถือว่าเป็นขั้นต่ำที่เหมาะสม


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. เชื่อมต่อกับ Foundry Local

`FoundryLocalManager` ดาวน์โหลดโมเดลหากจำเป็น, เริ่มต้นบริการในเครื่อง, และให้เราได้ **ปลายทางที่เข้ากันได้กับ OpenAI** จากนั้นเราจึงชี้ไปยัง SDK ของ OpenAI มาตรฐานที่นั่น คีย์ API เป็นเพียงตัวแทนในเครื่อง — ไม่มีข้อมูลรับรองบนคลาวด์เกี่ยวข้อง


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. เครื่องมือภายในเครื่อง (การดำเนินการไฟล์แบบแซนด์บ็อกซ์)

เราสร้างโปรเจ็กต์ตัวอย่างเล็กๆ บนดิสก์ จากนั้นกำหนดเครื่องมือที่จำกัดขอบเขตไว้ที่รากโปรเจ็กต์นั้น การตรวจสอบแซนด์บ็อกซ์ยังมีความสำคัญแม้ในเครื่อง: เครื่องมือที่อ่านเส้นทางใดๆ ได้จะทำงานด้วยสิทธิ์ของผู้ใช้ของคุณและสามารถแตะต้องสิ่งใดก็ได้ที่คุณแตะต้องได้


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. RAG ท้องถิ่นกับ Chroma

เราฝังชุดเอกสารย่อยขนาดเล็กไว้ในคอลเลกชัน Chroma ท้องถิ่น Chroma ทำงานในกระบวนการเดียวกันและจัดเก็บเวกเตอร์บนดิสก์ — ไม่มีเซิร์ฟเวอร์ ไม่มีคลาวด์ เครื่องมือ `search_docs` จะดึงเอกสารย่อยที่เกี่ยวข้องที่สุดสำหรับคำค้นหา


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. วงจรการเรียกใช้เครื่องมือ

ตอนนี้เราลงทะเบียนเครื่องมือกับโมเดลโดยใช้สกีมาเครื่องมือ OpenAI และรันวงจรการเรียกใช้เครื่องมือมาตรฐาน — โมเดลขอเครื่องมือ เราจะดำเนินการเครื่องมือนั้นในเครื่องของเรา ส่งผลลัพธ์กลับไป และทำซ้ำจนกว่าโมเดลจะให้คำตอบสุดท้าย การเรียกใช้ฟังก์ชันที่เชื่อถือได้ของ Qwen คือสิ่งที่ทำให้สิ่งนี้ทำงานได้บนอุปกรณ์


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. MCP ท้องถิ่น (ไม่บังคับ)

MCP คือช่องทางการขนส่ง ไม่ใช่บริการคลาวด์ — เซิร์ฟเวอร์ MCP สามารถทำงานเป็นกระบวนการท้องถิ่นผ่าน `stdio` ได้ เซลล์ด้านล่างแสดงวิธีที่คุณจะเชื่อมต่อกับเซิร์ฟเวอร์ MCP ท้องถิ่นหากคุณได้ตั้งค่าไว้ (เช่น เซิร์ฟเวอร์ระบบไฟล์) โดยจะข้ามไปอย่างเรียบร้อยเมื่อ `LOCAL_MCP_COMMAND` ไม่ได้ตั้งค่าไว้ ดังนั้นสมุดโน้ตยังสามารถทำงานได้ครบถ้วนแม้ไม่มีเซิร์ฟเวอร์นี้

หมายเหตุด้านความปลอดภัย: เซิร์ฟเวอร์ MCP ท้องถิ่นทำงานด้วยสิทธิ์ของผู้ใช้คุณ จำกัดขอบเขตให้กับไดเรกทอรีโครงการและตรวจสอบผลลัพธ์ก่อนที่จะดำเนินการใดๆ กับมัน


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## สรุป

คุณได้สร้างผู้ช่วยด้านวิศวกรรมที่ทำงานทั้งหมดบนเครื่องของคุณ:

- **Foundry Local** ให้บริการโมเดล **Qwen** ผ่านทาง endpoint ที่เข้ากันได้กับ OpenAI — ดังนั้นโค้ดตัวแทนจึงตรงกับบทเรียนบนคลาวด์
- **Sandboxed tools** ให้สิทธิ์ตัวแทนในการเข้าถึงไฟล์และวิเคราะห์โค้ดโดยไม่ต้องออกจากไดเรกทอรีโครงการ
- **Chroma** ให้บริการ **local RAG** สำหรับเอกสารประกอบ
- **Local MCP** แสดงวิธีการใช้ระบบ MCP ซ้ำในโหมดออฟไลน์

ไม่มีการใช้การคาดการณ์จากคลาวด์ในทุกช่วงเวลา

### ความท้าทาย

ขยายตัวแทนท้องถิ่นให้สามารถ:

1. **ทำงานกับเซิร์ฟเวอร์ MCP หลายตัว** — เชื่อมต่อเซิร์ฟเวอร์ระบบไฟล์และเซิร์ฟเวอร์ git และให้ตัวแทนเลือกใช้งานในระหว่างนั้น
2. **ใช้หน่วยความจำภายในเครื่อง** — บันทึกประวัติการสนทนาไว้บนดิสก์เพื่อให้ผู้ช่วยจำรอบก่อนหน้าได้แม้จะรีสตาร์ทโน้ตบุ๊ก
3. **รองรับการประสานงานตัวแทนหลายตัวในเครื่อง** — เพิ่มตัวแทนท้องถิ่นตัวที่สอง (เช่น ตัวตรวจสอบ) และให้ทั้งสองร่วมมือกันในงานหนึ่ง

ในบทเรียนถัดไปคุณจะได้เรียนรู้วิธีการรักษาความปลอดภัยกับตัวแทน AI ที่ถูก deploy


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
